# Chapter 10 — Basic Time Series
## University of Neuchâtel — Python for Econometrics

Topics:
1. Time series overview and notation
2. Static models
3. Finite Distributed Lag (FDL) models
4. Trends and seasonality
5. Serial correlation in residuals (Durbin-Watson)

**Format:** One cell per step. Manual first, statsmodels second, `np.isclose()`.

## 0 — Setup

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from statsmodels.stats.stattools import durbin_watson
from statsmodels.graphics.tsaplots import plot_acf
import matplotlib.pyplot as plt
import wooldridge as woo
import os

In [ ]:
chapter_dir = os.path.dirname(os.path.abspath('__file__'))
os.chdir(chapter_dir)
output_dir = os.path.join(chapter_dir, 'output')
os.makedirs(output_dir, exist_ok=True)
print('Working directory:', chapter_dir)

## 1 — Time Series Basics

**Key differences from cross-section:**
- Observations are ordered in time: $y_1, y_2, ..., y_T$
- Adjacent observations are often **correlated** (serial correlation)
- Series may have **trends** or **seasonality**

**Strict exogeneity** (TS.3): $E(u_t|X) = 0$ for all $t$ — stronger than cross-section assumption.

## 2 — Load Data: `phillips`

In [ ]:
df = woo.dataWoo('phillips')
print('Shape:', df.shape)
print(df[['year','inf','unem']].head(10))

In [ ]:
print('Summary stats:')
print(df[['inf','unem']].describe().round(3))

In [ ]:
fig, ax1 = plt.subplots(figsize=(10,4))
ax2 = ax1.twinx()
ax1.plot(df['year'], df['inf'], 'b-', label='Inflation', linewidth=2)
ax2.plot(df['year'], df['unem'], 'r--', label='Unemployment', linewidth=2)
ax1.set_xlabel('Year')
ax1.set_ylabel('Inflation (%)', color='blue')
ax2.set_ylabel('Unemployment (%)', color='red')
ax1.set_title('US Inflation and Unemployment 1948-1996')
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1+lines2, labels1+labels2, loc='upper right')
plt.tight_layout()
plt.savefig(os.path.join(output_dir,'fig_ch10_01_inf_unem.png'), dpi=150, bbox_inches='tight')
plt.show()

## 3 — Static Model: inf ~ unem

**Static model:** current value of $y$ depends on current values of $x$:

$$inf_t = \beta_0 + \beta_1 unem_t + u_t$$

This is the Phillips curve: inverse relationship between inflation and unemployment.

### Step 3.1 — Manual OLS

In [ ]:
n = len(df)
x = df['unem'].values
y = df['inf'].values
b1_man = np.sum((x-x.mean())*(y-y.mean())) / np.sum((x-x.mean())**2)
b0_man = y.mean() - b1_man * x.mean()
print(f'Static model (manual): inf = {b0_man:.4f} + {b1_man:.4f}*unem')

### Step 3.2 — Statsmodels OLS

In [ ]:
res_static = smf.ols('inf ~ unem', data=df).fit()
print(res_static.summary())

### Step 3.3 — Compare

In [ ]:
print(f'b0: manual={b0_man:.4f}, sm={res_static.params["Intercept"]:.4f}, match={np.isclose(b0_man,res_static.params["Intercept"])}')
print(f'b1: manual={b1_man:.4f}, sm={res_static.params["unem"]:.4f}, match={np.isclose(b1_man,res_static.params["unem"])}')

## 4 — Finite Distributed Lag (FDL) Model

**FDL model:** current and past values of $x$ affect $y$:

$$inf_t = \alpha_0 + \delta_0 unem_t + \delta_1 unem_{t-1} + \delta_2 unem_{t-2} + u_t$$

- $\delta_0$: **impact multiplier** (immediate effect)
- $\delta_0 + \delta_1 + \delta_2$: **long-run propensity (LRP)** (total long-run effect)

### Step 4.1 — Create lagged variables manually

In [ ]:
df['unem_lag1'] = df['unem'].shift(1)
df['unem_lag2'] = df['unem'].shift(2)
print('Dataset with lags (first 6 rows):')
print(df[['year','unem','unem_lag1','unem_lag2','inf']].head(6))
print('Note: first 2 rows have NaN (no lagged values available)')

In [ ]:
df_fdl = df.dropna(subset=['unem_lag1','unem_lag2'])
print(f'After dropping NaN: {len(df_fdl)} observations')

### Step 4.2 — Manual OLS for FDL model

In [ ]:
n_fdl = len(df_fdl)
X_fdl = np.column_stack([
    np.ones(n_fdl),
    df_fdl['unem'].values,
    df_fdl['unem_lag1'].values,
    df_fdl['unem_lag2'].values
])
y_fdl = df_fdl['inf'].values
beta_fdl = np.linalg.solve(X_fdl.T @ X_fdl, X_fdl.T @ y_fdl)
labels_fdl = ['const','unem','unem_lag1','unem_lag2']
print('FDL model (manual):')
for lbl, b in zip(labels_fdl, beta_fdl):
    print(f'  {lbl:12s}: {b:.4f}')

### Step 4.3 — Statsmodels FDL

In [ ]:
res_fdl = smf.ols('inf ~ unem + unem_lag1 + unem_lag2', data=df_fdl).fit()
print(res_fdl.summary())

### Step 4.4 — Compare FDL manual vs statsmodels

In [ ]:
sm_fdl = res_fdl.params
print('FDL comparison:')
for lbl, b_man in zip(labels_fdl, beta_fdl):
    b_sm = sm_fdl[lbl]
    match = np.isclose(b_man, b_sm)
    print(f'  {lbl:12s}: manual={b_man:.4f}, sm={b_sm:.4f}, match={match}')

### Step 4.5 — Long-run propensity (LRP)

In [ ]:
LRP_manual = beta_fdl[1] + beta_fdl[2] + beta_fdl[3]
LRP_sm     = res_fdl.params['unem'] + res_fdl.params['unem_lag1'] + res_fdl.params['unem_lag2']
print(f'LRP (manual):      {LRP_manual:.4f}')
print(f'LRP (statsmodels): {LRP_sm:.4f}')
print(f'Match: {np.isclose(LRP_manual, LRP_sm)}')
print(f'Interpretation: A permanent 1pp rise in unem eventually changes inf by {LRP_manual:.4f}pp')

## 5 — Trend Regression

**When to add a trend:** If both $y$ and $x$ have time trends, the regression may be
spurious unless we control for trend.

**Linear trend:** add $t = 1, 2, ..., T$

In [ ]:
df_fdl = df_fdl.copy()
df_fdl['t'] = np.arange(1, len(df_fdl)+1)
print('Trend variable added (first 5 rows):')
print(df_fdl[['year','unem','inf','t']].head())

In [ ]:
n_tr = len(df_fdl)
X_tr = np.column_stack([
    np.ones(n_tr),
    df_fdl['unem'].values,
    df_fdl['t'].values
])
y_tr = df_fdl['inf'].values
beta_tr = np.linalg.solve(X_tr.T @ X_tr, X_tr.T @ y_tr)
print('Model with trend (manual):')
for lbl, b in zip(['const','unem','t'], beta_tr):
    print(f'  {lbl:6s}: {b:.4f}')

In [ ]:
res_trend = smf.ols('inf ~ unem + t', data=df_fdl).fit()
sm_tr = res_trend.params
print('Trend model comparison:')
for lbl_m, lbl_s, b_man in zip(['const','unem','t'], ['Intercept','unem','t'], beta_tr):
    b_sm = sm_tr[lbl_s]
    print(f'  {lbl_s:12s}: manual={b_man:.4f}, sm={b_sm:.4f}, match={np.isclose(b_man,b_sm)}')

## 6 — Serial Correlation in Residuals

In [ ]:
e_static = res_static.resid.values
fig, ax = plt.subplots(figsize=(10,4))
ax.plot(df['year'], e_static, marker='o', markersize=3, linewidth=1)
ax.axhline(0, color='red', linestyle='--')
ax.set_xlabel('Year')
ax.set_ylabel('Residual')
ax.set_title('Static model residuals over time')
plt.tight_layout()
plt.savefig(os.path.join(output_dir,'fig_ch10_02_ts_residuals.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Pattern in residuals → serial correlation')

In [ ]:
# Durbin-Watson
DW_manual = np.sum(np.diff(e_static)**2) / np.sum(e_static**2)
DW_sm = durbin_watson(e_static)
print(f'Durbin-Watson (manual):      {DW_manual:.4f}')
print(f'Durbin-Watson (statsmodels): {DW_sm:.4f}')
print(f'Match: {np.isclose(DW_manual, DW_sm)}')
if DW_sm < 1.5:
    print('DW < 1.5: positive serial correlation in residuals')

## 7 — Summary Table

In [ ]:
sumtab = pd.DataFrame({
    'Model': ['Static','FDL (unem only)','With trend'],
    'b_unem': [res_static.params['unem'], res_fdl.params['unem'], res_trend.params['unem']],
    'R2':     [res_static.rsquared, res_fdl.rsquared, res_trend.rsquared]
})
print(sumtab.round(4).to_string(index=False))
sumtab.to_csv(os.path.join(output_dir,'table_ch10_ts_models.csv'), index=False)
print('Saved table_ch10_ts_models.csv')